In [4]:
import itertools
import re
import multiprocess
import numpy as np
import pandas as pd
from pymatgen.core import Composition # check if this could be changed to smact.Composition
import matplotlib.pyplot as plt

import smact
from smact import Element, Species
from smact.oxidation_states import Oxidation_state_probability_finder
from smact.screening import pauling_test, neutral_ratios, smact_validity, smact_filter

# Initialize the oxidation state probability finder
ox_prob_finder = Oxidation_state_probability_finder()

# Define the chemical system and parameters
CHEM_ELEMENTS = ["Cu", "Ti", "O"]
MAX_STOICH = 8  # maximum stoichiometric coefficient
oxidation_states_set = "icsd24"  # ICSd (2024) dataset for oxidation states

# Create smact Elements
elements = [Element(el) for el in CHEM_ELEMENTS]

# Generate all possible compositions (up to MAX_STOICH)
all_compositions = []
for stoich_tuple in itertools.product(range(0, MAX_STOICH), repeat=len(elements)):
    comp_dict = {el.symbol: amt for el, amt in zip(elements, stoich_tuple)}

    comp = Composition(comp_dict)
    # Filter trivial invalid compositions if desired
    # (e.g., exclude ones that are known improbable by smact_validity)
    if smact_validity(comp, oxidation_states_set=oxidation_states_set):
        all_compositions.append(comp)

print(f"Number of initial compositions (raw): {len(all_compositions)}")


# Detailed filtering using SMACT oxidation states and electronegativity checks
#
# We'll use smact_filter to find all balanced compositions including their
# oxidation states and ratios. smact_filter returns:
# [( (element_symbols), (ox_states), (ratios) ), ...]

# NOTE: cheeck how smact_filter works compared to smact_validity - to the notebook create an example of advaced vs normal usage for the course

# smact_filter needs a tuple of Elements. We can supply just elements and a threshold.
filtered_results = smact_filter(tuple(elements), threshold=MAX_STOICH, species_unique=True, oxidation_states_set=oxidation_states_set)

# Extract only those compositions that made it through the filter
allowed_comps = []
for entry in filtered_results:
    # entry format: ( (symbols), (ox_states), (ratios) )
    syms, ox_states, ratios = entry
    c = Composition({s: r for s, r in zip(syms, ratios)}).reduced_composition
    if c not in allowed_comps:
        allowed_comps.append(c)

print(f"Number of allowed compositions after SMACT filtering: {len(allowed_comps)}")


ValueError: If using all scalar values, you must pass an index

In [3]:

# Compute oxidation state probabilities for each allowed composition
#
# To use the `compound_probability` method, we need a list of Species with
# assigned oxidation states. `smact_filter` gives us possible oxidation states,
# but we need to pick a representative from the filtered_results.
#
# For simplicity, we’ll compute probabilities for all sets generated. In practice,
# you could also just pick the best scoring oxidation states combination.

all_entries = []
for entry in filtered_results:
    syms, ox_states, ratios = entry
    # Construct a list of smact.Species objects
    species_list = [Species(s, o) for s, o in zip(syms, ox_states)]
    # The `compound_probability` function can take a list of Species
    # By default, `ignore_stoichiometry=True` averages probabilities over species pairs.
    comp_prob = ox_prob_finder.compound_probability(species_list, ignore_stoichiometry=True)

    # Build a readable formula
    c = Composition({s: r for s, r in zip(syms, ratios)})
    pretty_formula = c.reduced_formula

    # Store
    all_entries.append({
        "formula_pretty": pretty_formula,
        "symbols": syms,
        "oxidation_states": ox_states,
        "ratios": ratios,
        "compound_probability": comp_prob
    })

df = pd.DataFrame(all_entries)


NameError: One or both of [Cu1, O-1] are not in the probability table.

In [4]:

# Inspect and analyze results
print(f"Number of compositions in DataFrame: {len(df)}")
print("Sample entries:")
print(df.head())

# Count how many have non-zero probabilities
non_zero_count = np.count_nonzero(df["compound_probability"].to_numpy() > 0)
print(
    f"The filtered SMACT search space produced {len(df)} compositions of which {non_zero_count} had a non-zero compound probability."
)

# Optional: Further filtering by probability
#
# You might want to only keep those compositions with a probability above some
# threshold. For example:
threshold_prob = 0.1
df_high_prob = df[df["compound_probability"] > threshold_prob]
print(f"Number of compositions with probability > {threshold_prob}: {len(df_high_prob)}")

# Now df_high_prob contains the subset of compositions with higher likelihoods

# You could proceed with further steps:
# - Visualize these compositions on a ternary diagram (Cu-Ti-O).
# - Query Materials Project for stability.
# - Perform MACE-MP pre-screening, etc.

print("Filtering and probability computations complete.")


NameError: name 'df' is not defined